# Train the cartoon colorizer on Colab

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

By default it trains on public-domain Max Fleischer *Superman* shorts (1941–43) from the Internet Archive, holding one episode out for testing. To use your own clips instead, set `USE_DRIVE = True` and point `MY_CLIPS` at a Drive folder.

In [ ]:
!git clone -q https://github.com/ArunaSamhitha/Cartoon-Colorization-Using-CycleGAN.git
%cd Cartoon-Colorization-Using-CycleGAN
!pip install -q -r requirements.txt
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU')

In [ ]:
# ---- Settings ----
USE_DRIVE = False                     # True: read clips from Drive and save the run there
MY_CLIPS = '/content/drive/MyDrive/cartoon_clips'
IMAGE_SIZE, N_BLOCKS, BATCH, EPOCHS, DECAY_START = 128, 6, 4, 60, 30
FRAME_EVERY = 24                      # keep 1 frame per second of video

OUT = '/content/drive/MyDrive/cyclegan_runs/exp1' if USE_DRIVE else '/content/runs/exp1'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

## 1. Get clips and extract frames

In [ ]:
import os, urllib.parse, subprocess
os.makedirs('clips/train', exist_ok=True); os.makedirs('clips/test', exist_ok=True)
BASE = 'https://archive.org/download/fleischer-superman/'
TRAIN = ["Max Fleischer's Superman\uff1a The Mechanical Monsters (1941; HD).mp4",
         "Max Fleischer's Superman\uff1a Billion Dollar Limited (1942; HD).mp4",
         "Max Fleischer's Superman\uff1a The Arctic Giant (1942; HD).mp4",
         "Max Fleischer's Superman\uff1a The Bulleteers (1942; HD).mp4",
         "Max Fleischer's Superman\uff1a The Magnetic Telescope (1942; HD).mp4",
         "Max Fleischer's Superman\uff1a The Electric Earthquake (1942; HD).mp4"]
TEST = ["Max Fleischer's Superman\uff1a Volcano (1942; HD).mp4"]

if USE_DRIVE:
    train_src, test_clip = MY_CLIPS, None
else:
    for name, folder in [(n, 'clips/train') for n in TRAIN] + [(n, 'clips/test') for n in TEST]:
        dst = f"{folder}/{len(os.listdir(folder))}.mp4"
        print('Downloading', name)
        subprocess.run(['wget', '-q', '-O', dst, BASE + urllib.parse.quote(name)], check=True)
    train_src = 'clips/train'
    test_clip = 'clips/test/0.mp4'
    # A 30-second black-and-white test clip the model has never seen
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-ss', '60', '-t', '30', '-i', test_clip,
                    '-vf', 'format=gray', 'clips/test_bw_30s.mp4'], check=True)

!python scripts/extract_frames.py --videos "$train_src" --out data/frames --every $FRAME_EVERY
if test_clip:
    !python scripts/extract_frames.py --videos "$test_clip" --out data/heldout --every 48

## 2. Train

About 1–1.5 hours on a T4 with the default settings. A sample grid is saved every epoch. If the session disconnects, re-run with `--resume "$OUT/checkpoints/last.pt"`.

In [ ]:
!python train.py --frames data/frames --out "$OUT" --image-size $IMAGE_SIZE --n-blocks $N_BLOCKS --batch-size $BATCH --epochs $EPOCHS --decay-start $DECAY_START

## 3. Look at progress

In [ ]:
import glob
from IPython.display import Image, display
last = sorted(glob.glob(f'{OUT}/samples/*.png'))[-1]
print(last); display(Image(last))

## 4. Results on held-out frames, and video speed

In [ ]:
!python scripts/make_results.py --ckpt "$OUT/checkpoints/colorizer.pt" --frames data/heldout --out assets/results.png --n 4
display(Image('assets/results.png'))
if os.path.exists('clips/test_bw_30s.mp4'):
    !python colorize.py --ckpt "$OUT/checkpoints/colorizer.pt" --input clips/test_bw_30s.mp4 --output /content/test_colorized.mp4

## 5. Summary and downloads

In [ ]:
import json
hist = [json.loads(l) for l in open(f'{OUT}/history.json')]
print('RUN SUMMARY', json.dumps({'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
      'train_frames': len(os.listdir('data/frames')), 'epochs': len(hist), 'image_size': IMAGE_SIZE,
      'n_blocks': N_BLOCKS, 'total_train_minutes': round(sum(h['seconds'] for h in hist) / 60, 1),
      'final_epoch': hist[-1]}))
from google.colab import files
files.download('assets/results.png')
files.download(f'{OUT}/history.json')